# Day 3 & 4: Advanced Analytics, Customer RFM Segmentation & ML Delay Prediction

**Author:** Salma El Moudi  
**Project:** E-Commerce Operations & Customer Intelligence  

This notebook covers:
1. **Recency, Frequency, Monetary (RFM)** Customer Segmentation
2. **Statistical Hypothesis Testing:** Testing whether delivery delays significantly affect customer review ratings (Mann-Whitney U Test)
3. **Predictive Machine Learning:** Logistic Regression to predict late orders at purchase time
4. Feature importance extraction for operational business recommendations

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# Load cleaned data
orders = pd.read_csv('../data/cleaned/orders_clean.csv')
order_items = pd.read_csv('../data/cleaned/order_items_clean.csv')
customers = pd.read_csv('../data/cleaned/customers_clean.csv')
reviews = pd.read_csv('../data/cleaned/reviews_clean.csv')
products = pd.read_csv('../data/cleaned/products_clean.csv')
sellers = pd.read_csv('../data/cleaned/sellers_clean.csv')

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
print(f"Loaded {len(orders):,} orders successfully.")

## 1. RFM Customer Segmentation

We calculate Recency, Frequency, and Monetary value for each unique customer (`customer_unique_id`).

In [2]:
ref_date = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

df_rfm = (
    orders.merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id', how='left')
    .merge(order_items[['order_id', 'price', 'freight_value']], on='order_id', how='left')
)
df_rfm = df_rfm[df_rfm['order_status'].isin(['delivered', 'shipped', 'invoiced'])]
df_rfm['order_value'] = df_rfm['price'] + df_rfm['freight_value']

rfm = (
    df_rfm.groupby('customer_unique_id')
    .agg(
        last_purchase=('order_purchase_timestamp', 'max'),
        frequency=('order_id', 'nunique'),
        monetary=('order_value', 'sum'),
        customer_state=('customer_state', 'first')
    )
    .reset_index()
)
rfm['recency'] = (ref_date - rfm['last_purchase']).dt.days

# Score 1 to 4
rfm['r_score'] = pd.qcut(rfm['recency'], 4, labels=[4, 3, 2, 1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], 4, labels=[1, 2, 3, 4]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

def segment_label(row):
    if row['rfm_score'] >= 10: return 'Champions'
    elif row['rfm_score'] >= 8: return 'Loyal Customers'
    elif row['rfm_score'] >= 6: return 'Potential Loyalists'
    elif row['r_score'] >= 3: return 'Recent Customers'
    elif row['rfm_score'] >= 4: return 'At Risk'
    else: return 'Lost Customers'

rfm['segment'] = rfm.apply(segment_label, axis=1)
print("RFM Segments distribution:")
print(rfm['segment'].value_counts())

## 2. Statistical Hypothesis Testing: Delivery Delays vs Customer Review Scores

**Hypothesis:** Orders delivered past their estimated delivery date (`is_late = 1`) receive statistically lower customer review ratings.

In [3]:
test_df = orders.merge(reviews[['order_id', 'review_score']], on='order_id', how='inner')
test_df = test_df[test_df['order_status'] == 'delivered'].dropna(subset=['review_score', 'is_late'])

on_time = test_df[test_df['is_late'] == 0]['review_score'].astype(float)
late = test_df[test_df['is_late'] == 1]['review_score'].astype(float)

u_stat, p_val = stats.mannwhitneyu(on_time, late, alternative='greater')
print(f"Average review score (On-Time): {on_time.mean():.3f}")
print(f"Average review score (Late):    {late.mean():.3f}")
print(f"Mann-Whitney U statistic: {u_stat:,.0f}, p-value: {p_val}")
print(f"Conclusion: {'Statistically significant difference (p < 0.05)' if p_val < 0.05 else 'No difference'}")

## 3. Machine Learning: Delivery Delay Predictor (Logistic Regression)

We train an interpretable classifier to predict whether an order will breach its delivery SLA based on properties known at purchase time.

In [4]:
orders_ml = orders.copy()
orders_ml['order_estimated_delivery_date'] = pd.to_datetime(orders_ml['order_estimated_delivery_date'])
orders_ml['estimated_delivery_days'] = (
    orders_ml['order_estimated_delivery_date'] - orders_ml['order_purchase_timestamp']
).dt.days

df_ml = (
    orders_ml[['order_id', 'is_late', 'estimated_delivery_days', 'customer_id']]
    .merge(order_items[['order_id', 'product_id', 'seller_id', 'price', 'freight_value']], on='order_id', how='left')
    .merge(products[['product_id', 'product_weight_g']], on='product_id', how='left')
    .merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
    .merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')
)
df_ml = df_ml.dropna(subset=['is_late', 'product_weight_g', 'freight_value', 'price', 'estimated_delivery_days'])
df_ml = df_ml[df_ml['estimated_delivery_days'] > 0]
df_ml['is_late'] = df_ml['is_late'].astype(int)

top_states = df_ml['customer_state'].value_counts().nlargest(5).index
df_ml['cust_state_enc'] = df_ml['customer_state'].apply(lambda x: x if x in top_states else 'other')
df_ml['sell_state_enc'] = df_ml['seller_state'].apply(lambda x: x if x in top_states else 'other')
df_ml = pd.get_dummies(df_ml, columns=['cust_state_enc', 'sell_state_enc'], drop_first=True)

feature_cols = (
    ['product_weight_g', 'freight_value', 'price', 'estimated_delivery_days']
    + [c for c in df_ml.columns if c.startswith('cust_state_enc_') or c.startswith('sell_state_enc_')]
)
X = df_ml[feature_cols].fillna(0)
y = df_ml['is_late']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)
y_prob = model.predict_proba(X_test_sc)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Feature Importance
feat_importance = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': model.coef_[0].round(4),
    'abs_coef': np.abs(model.coef_[0]).round(4)
}).sort_values('abs_coef', ascending=False)

print("\nTop Predictors of Late Delivery:")
print(feat_importance.head(8).to_string(index=False))